In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuredatabricks_catalog.ingest.sales_tbl  
(
    sale_id INT,
    store_id INT,
    sale_qty INT,
    amount FLOAT
)
USING DELTA
LOCATION "abfss://raw@azuredatabricksansh.dfs.core.windows.net/optimize/sales_tbl"
TBLPROPERTIES (
    'delta.enableDeletionVectors' = 'false'
)

In [0]:
%sql
INSERT INTO azuredatabricks_catalog.ingest.sales_tbl
VALUES (6, 1, 10, 100),
       (7, 1, 20, 200),
       (8, 2, 30, 300),
       (9, 2, 40, 400),
       (10, 3, 50, 500);

In [0]:
%sql
UPDATE azuredatabricks_catalog.ingest.sales_tbl
SET sale_qty = sale_qty + 100
WHERE sale_id=1;

In [0]:
%sql
DELETE FROM azuredatabricks_catalog.ingest.sales_tbl
WHERE sale_id=2;

In [0]:
%sql
DESCRIBE HISTORY azuredatabricks_catalog.ingest.sales_tbl

In [0]:
%sql
SELECT * FROM azuredatabricks_catalog.ingest.sales_tbl VERSION AS OF 3

In [0]:
%sql
RESTORE azuredatabricks_catalog.ingest.sales_tbl TO VERSION AS OF 3

In [0]:
%sql
DESCRIBE HISTORY azuredatabricks_catalog.ingest.sales_tbl

# Deletion Vector

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuredatabricks_catalog.ingest.sales_tbl2  
(
    sale_id INT,
    store_id INT,
    sale_qty INT,
    amount FLOAT
)
USING DELTA
LOCATION "abfss://raw@azuredatabricksansh.dfs.core.windows.net/optimize/sales_tbl2"


In [0]:
%sql
INSERT INTO azuredatabricks_catalog.ingest.sales_tbl2
VALUES (6, 1, 10, 100),
       (7, 1, 20, 200),
       (8, 2, 30, 300),
       (9, 2, 40, 400),
       (10, 3, 50, 500);

In [0]:
%sql
UPDATE azuredatabricks_catalog.ingest.sales_tbl2
SET sale_qty = sale_qty + 100
WHERE sale_id=6;

In [0]:
df = spark.read.format("parquet")\
        .load("abfss://raw@azuredatabricksansh.dfs.core.windows.net/optimize/part-00000-21e6dbda-eb74-4004-9f5f-33dc9ffeb322.c000.snappy.parquet")
display(df)

# Partitioning

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuredatabricks_catalog.ingest.sales_part 
(
    sale_id INT,
    store_id INT,
    sale_qty INT,
    amount FLOAT,
    year INT
)
USING DELTA
LOCATION "abfss://raw@azuredatabricksansh.dfs.core.windows.net/optimize/sales_part"
PARTITIONED BY (year)

In [0]:
%sql
INSERT INTO azuredatabricks_catalog.ingest.sales_part
VALUES (11, 1, 10, 100, 2022),
       (12, 1, 20, 200, 2022),
       (13, 2, 30, 300, 2022),
       (14, 2, 40, 400, 2022),
       (15, 3, 50, 500, 2026),
       (16, 3, 60, 600, 2026),
       (17, 4, 70, 700, 2026),
       (18, 4, 80, 800, 2026);
    

In [0]:
%sql
SELECT * FROM azuredatabricks_catalog.ingest.sales_part
WHERE year = 2026

# OPTIMIZE

In [0]:
%sql
OPTIMIZE azuredatabricks_catalog.ingest.sales_part ZORDER BY (store_id)
    


In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuredatabricks_catalog.ingest.sales_clustered
(
    sale_id INT,
    store_id INT,
    sale_qty INT,
    amount FLOAT,
    year INT
)
USING DELTA
LOCATION "abfss://raw@azuredatabricksansh.dfs.core.windows.net/optimize/sales_clustered"
CLUSTER BY (sale_id)

In [0]:
%sql
INSERT INTO azuredatabricks_catalog.ingest.sales_clustered
VALUES (11, 1, 10, 100, 2022),
       (12, 1, 20, 200, 2022),
       (13, 2, 30, 300, 2022),
       (14, 2, 40, 400, 2022),
       (15, 3, 50, 500, 2026),
       (16, 3, 60, 600, 2026),
       (17, 4, 70, 700, 2026),
       (18, 4, 80, 800, 2026);
    

In [0]:
%sql
DESCRIBE EXTENDED azuredatabricks_catalog.ingest.sales_clustered

In [0]:
%sql
ALTER TABLE azuredatabricks_catalog.ingest.sales_part
REPLACE PARTITIONED BY WITH CLUSTER BY AUTO

In [0]:
%sql
ALTER TABLE azuredatabricks_catalog.ingest.sales_part
CLUSTER BY NONE

In [0]:
%sql
DESCRIBE EXTENDED azuredatabricks_catalog.ingest.sales_part

In [0]:
%sql
OPTIMIZE azuredatabricks_catalog.ingest.sales_part;